# 01 — Business Understanding (CRISP-DM)

**Tujuan:** Menetapkan masalah bisnis, tujuan SMART, metrik keberhasilan, asumsi/batasan, risiko/etika, nilai bisnis, dan pemetaan stakeholder–data untuk proyek estimasi kalori makanan Indonesia.

**Daftar Tugas:**
- [Progress 1 – Capaian 1] Judul spesifik dan ruang lingkup tepat
- [Progress 1 – Capaian 2] Judul selaras tujuan dan metrik
- [Progress 1 – Capaian 3] Pernyataan masalah dan stakeholder
- [Progress 1 – Capaian 4] Tujuan SMART
- [Progress 1 – Capaian 5] Metrik keberhasilan
- [Progress 1 – Capaian 6] Asumsi dan batasan
- [Progress 1 – Capaian 7] Risiko dan etika + mitigasi
- [Progress 1 – Capaian 8] Nilai bisnis / use-case
- [Progress 1 – Capaian 9] Pemetaan Stakeholder – Data


In [1]:

from pathlib import Path
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().resolve().parents[0]
RAW = ROOT / 'data' / 'raw' / 'nutrition.csv'
PROC = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs'
FIG = ROOT / 'reports' / 'figures'
for p in [PROC, OUT, FIG]:
    p.mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)

df = pd.read_csv(RAW)
print(df.shape)
df.head()


ROOT: C:\.My Data\Kuliah\semester 5\Data_Science_TugasAkhir\uts-data-science-gizi
(1346, 7)


,id,calories,proteins,fat,carbohydrate,name,image
0,1,280.0,9.2,28.4,0.0,Abon,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...
1,2,513.0,23.7,37.0,21.3,Abon haruwan,https://img-global.cpcdn.com/recipes/cbf330fbd...
2,3,0.0,0.0,0.2,0.0,Agar-agar,https://res.cloudinary.com/dk0z4ums3/image/upl...
3,4,45.0,1.1,0.4,10.8,Akar tonjong segar,https://images.tokopedia.net/img/cache/200-squ...
4,5,37.0,4.4,0.5,3.8,Aletoge segar,https://nilaigizi.com/assets/images/produk/pro...


## [Progress 1 – Capaian 1] Judul spesifik dan ruang lingkup tepat

**Judul proyek:** *Estimasi Kandungan Kalori Makanan Indonesia dari Protein, Lemak, dan Karbohidrat Menggunakan Regresi Linear dengan Pembanding SVR*

**Ruang lingkup:**
- **Objek:** makanan/minuman Indonesia pada dataset `nutrition.csv`.
- **Target:** `calories` (kkal; satuan mengikuti dataset, belum terverifikasi dari metadata resmi).
- **Metode utama:** Regresi Linear (OLS, `sklearn.linear_model.LinearRegression`) + inferensi/diagnostik `statsmodels.OLS`.
- **Pembanding:** SVR (`sklearn.svm.SVR`) — SVM versi regresi, bukan SVC.
- **Fitur dasar:** `proteins`, `fat`, `carbohydrate`.
- **Batas data:** 1.346 baris × 7 kolom (`id, calories, proteins, fat, carbohydrate, name, image`); data statis/cross-sectional, bukan deret waktu.


In [2]:

n_rows, n_cols = df.shape
print({'n_rows': n_rows, 'n_cols': n_cols, 'columns': df.columns.tolist()})
assert n_rows == 1346 and n_cols == 7
assert df.columns.tolist() == ['id','calories','proteins','fat','carbohydrate','name','image']


{'n_rows': 1346, 'n_cols': 7, 'columns': ['id', 'calories', 'proteins', 'fat', 'carbohydrate', 'name', 'image']}


## [Progress 1 – Capaian 2] Judul selaras tujuan dan metrik

| Kata kunci di judul | Tujuan terkait | Metrik/artefak |
|---|---|---|
| Estimasi kandungan kalori | Memprediksi nilai kalori numerik | R², MAE, RMSE |
| Makanan Indonesia | Konteks domain pangan Indonesia | Data `nutrition.csv`, analisis representativitas |
| Protein, lemak, karbohidrat | Menjelaskan kontribusi makronutrien | Koefisien OLS, VIF, korelasi |
| Regresi Linear | Model utama yang interpretabel | R², Adjusted R², MAE, RMSE, diagnostik OLS |
| Pembanding SVR | Membandingkan model linear vs non-linear | Tabel perbandingan metrik yang sama |

Konsistensi: judul menyebut *kalori* + *regresi* → target numerik kontinu → metrik regresi (R²/MAE/RMSE), bukan metrik klasifikasi.


## [Progress 1 – Capaian 3] Pernyataan masalah dan stakeholder

### Pernyataan masalah
Data kalori pada database pangan dapat kosong, tidak konsisten, atau sulit diverifikasi saat entri baru ditambahkan. Pengguna aplikasi gizi dan pengelola database membutuhkan estimasi kalori cepat berbasis komposisi makronutrien (protein, lemak, karbohidrat) yang **dapat dijelaskan** dan **dievaluasi secara kuantitatif**. Model harus membantu estimasi/validasi, bukan menggantikan sumber gizi resmi atau tenaga kesehatan.

### Stakeholder
| Stakeholder | Peran/kepentingan |
|---|---|
| Pengguna aplikasi gizi | Mendapat estimasi kalori cepat saat data belum tersedia |
| Ahli gizi / penyusun menu | Memvalidasi kewajaran estimasi dan batas pakai |
| Pengelola database pangan | Mendeteksi entri kalori yang tidak konsisten |
| Pengembang aplikasi | Mengintegrasikan model ke fitur estimasi/validasi |
| Peneliti / dosen penilai | Menilai metodologi, reproduktibilitas, dan etika |


## [Progress 1 – Capaian 4] Tujuan SMART

Target berikut ditetapkan **sebelum melihat data test** dan tidak boleh diubah setelah melihat hasil.

| Tujuan | S | M | A | R | T |
|---|---|---|---|---|---|
| T1: Akurasi Linear Regression | Memprediksi kalori makanan Indonesia | R² test ≥ 0,80; MAE test ≤ 45 kkal; RMSE test ≤ 70 kkal | Menggunakan 3 fitur makro + pipeline reproducible | Mendukung estimasi kalori cepat | Sesuai jadwal Progress 4 mata kuliah |
| T2: Perbandingan model | Membandingkan Linear Regression vs SVR | Metrik sama (R², Adj R², MAE, RMSE) pada split yang sama | SVR dengan scaling dan tuning CV pada train | Memberi dasar pemilihan model | Sesuai jadwal Progress 4 mata kuliah |
| T3: Kualitas & interpretabilitas | Menyampaikan hasil yang bisa dijelaskan | Ada koefisien OLS, diagnostik asumsi, analisis error ≥10 kasus | Memakai statsmodels + audit data | Mendukung keputusan stakeholder | Sesuai jadwal Progress 4 mata kuliah |


## [Progress 1 – Capaian 5] Metrik keberhasilan

### Rumus
- **R²** = $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ — proporsi variasi target yang dijelaskan model.
- **Adjusted R²** = $1 - (1-R^2)\frac{n-1}{n-p-1}$ — R² yang dikoreksi jumlah fitur.
- **MAE** = $\frac{1}{n}\sum |y_i - \hat{y}_i|$ — rata-rata selisih absolut (kkal).
- **RMSE** = $\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$ — akar rata-rata kuadrat error (kkal), lebih sensitif ke error besar.
- **MAPE** = $\frac{100\%}{n}\sum |\frac{y_i-\hat{y}_i}{y_i}|$ — pelengkap saja; **tidak utama** karena ada nilai `calories=0` dan nilai kecil membuat MAPE tidak stabil/terdefinisi.

### Ambang target (ditetapkan sebelum melihat test)
| Metrik | Target | Makna praktis |
|---|---:|---|
| R² test | ≥ 0,80 | Model menjelaskan ≥80% variasi kalori test |
| MAE test | ≤ 45 kkal | Rata-rata meleset ≤45 kkal |
| RMSE test | ≤ 70 kkal | Error besar dihukum; target RMSE ≤70 kkal |


## [Progress 1 – Capaian 6] Asumsi dan batasan

### Asumsi
- Nilai gizi pada baris dataset merepresentasikan komposisi yang sebanding (satuan mengikuti dataset; belum terverifikasi).
- Hubungan makronutrien utama (protein, lemak, karbohidrat) dengan kalori mendekati linear secara dominan (Atwater: ~4/9/4 kkal/g).
- Data cukup mewakili variasi makanan Indonesia untuk tujuan pembelajaran/prototipe.

### Batasan
- Tidak ada fitur serat, air, alkohol, gula, atau porsi/berat sajian.
- Sumber asli dan lisensi `nutrition.csv` belum terverifikasi dari metadata repo.
- Ada anomali input (makronutrien > 100 g, duplikat nama, kalori 0, outlier ekstrem).
- Kolom `metode_olah`/`kelompok_pangan` (jika dipakai di ablation) adalah heuristik dari nama, bukan kategori resmi.
- Model tidak boleh dipakai sebagai saran medis/diet klinis.


## [Progress 1 – Capaian 7] Risiko dan etika + mitigasi

| Risiko | Dampak | Mitigasi |
|---|---|---|
| Bias representasi makanan | Model kurang akurat untuk makanan daerah/tipe yang kurang terwakili | Analisis representativitas, laporkan kelompok dominan/minor, saran penambahan data |
| Estimasi dipakai untuk keputusan medis/diet | Risiko kesehatan jika salah | Disclaimer jelas: bukan pengganti data resmi/tenaga gizi; tampilkan MAE/RMSE |
| Kualitas data salah input | Model belajar pola salah | Audit kualitas, aturan cleaning transparan, log baris terdampak |
| Lisensi/sumber data tidak jelas | Risiko kepatuhan penggunaan | Catat status "belum terverifikasi", jangan klaim lisensi; hindari pakai URL gambar |
| Kebocoran target | Metrik terlalu optimistis | Larang `calories`, `kategori_kalori`, `energi_atwater` sebagai fitur utama; split sebelum fit transform |


## [Progress 1 – Capaian 8] Nilai bisnis / use-case

| Use-case | Pengguna | Nilai | Cara ukur |
|---|---|---|---|
| Pengisian otomatis kalori kosong | Pengelola database pangan | Mengurangi entri kosong/inkonsisten | % entri kosong terisi, MAE pada sampel verifikasi |
| Validasi input data gizi | Admin database / QA | Mendeteksi kalori yang jauh dari estimasi makro | Jumlah anomali terdeteksi, precision review manual |
| Estimasi cepat di aplikasi pelacak kalori | Pengguna aplikasi gizi | Memberi estimasi awal sebelum verifikasi resmi | Latensi prediksi, MAE, feedback pengguna |


## [Progress 1 – Capaian 9] Pemetaan Stakeholder – Data

| Stakeholder | Kebutuhan | Kolom/data relevan | Keputusan yang didukung |
|---|---|---|---|
| Pengguna aplikasi gizi | Estimasi kalori cepat | `proteins,fat,carbohydrate` → `calories` | Menampilkan estimasi awal kalori |
| Ahli gizi / penyusun menu | Kewajaran nilai gizi | `name, calories, proteins, fat, carbohydrate`, audit Atwater | Menerima/menolak estimasi, review anomali |
| Pengelola database pangan | Deteksi inkonsistensi | Semua kolom numerik, `name`, selisih Atwater | Prioritas pembersihan/validasi data |
| Pengembang aplikasi | Integrasi model | Fitur model, artefak pipeline, metrik | Memilih model untuk produksi/prototipe |
| Peneliti / dosen | Reproduktibilitas & evaluasi | `outputs/`, `reports/figures/`, notebook | Penilaian capaian proyek |


In [3]:

bu = pd.DataFrame({
    'item':['judul','n_rows','n_cols','target','fitur_utama','split','random_state','r2_target','mae_target','rmse_target'],
    'nilai':['Estimasi Kandungan Kalori Makanan Indonesia dari Protein, Lemak, dan Karbohidrat Menggunakan Regresi Linear dengan Pembanding SVR',
             int(df.shape[0]), int(df.shape[1]), 'calories', 'proteins;fat;carbohydrate', '80/20 stratified by calories bins', 42, 0.80, 45, 70]
})
bu.to_csv(OUT/'business_understanding_summary.csv', index=False)
bu


,item,nilai
0,judul,Estimasi Kandungan Kalori Makanan Indonesia da...
1,n_rows,1346
2,n_cols,7
3,target,calories
4,fitur_utama,proteins;fat;carbohydrate
5,split,80/20 stratified by calories bins
6,random_state,42
7,r2_target,0.8
8,mae_target,45
9,rmse_target,70


**Temuan:** Ruang lingkup proyek ditetapkan sebagai regresi kalori makanan Indonesia menggunakan 3 fitur makronutrien utama, dengan Linear Regression sebagai model utama dan SVR sebagai pembanding. Target SMART ditetapkan sebelum evaluasi test untuk mencegah perubahan target setelah melihat hasil.
